# ECON 4370 — Homework 3
## Build Your Own Economic AI Agent

**Posted:** TBD  
**Due:** TBD (11:59 PM, Blackboard Ultra)

> **How to use this notebook:**
> - Replace all **TODO** text with your work.
> - Keep written answers in **Markdown** cells.
> - Make sure the notebook runs **top → bottom** without errors before submitting.
> - Submit: this `.ipynb` file + a PDF export via Blackboard Ultra.

---

### What this assignment is about

In Lecture 8 you studied two agent versions: a structured baseline and a more autonomous intermediate agent. In this homework you will build your **own** AI economic data agent from scratch — not a copy of the lecture code.

You will:
1. Choose an original research question and assemble the data pipeline to answer it
2. Extend the agent with a tool that was **not** in lecture (regression or a second API source)
3. Run and document three original queries through your agent
4. Critically evaluate where your agent succeeds — and where it fails

**AI Policy Reminder:** You may use AI tools to help you understand concepts, debug code, or improve writing. You may **not** have AI generate the final submitted notebook on your behalf. You must be able to explain every line of your code if asked.

---

## Student Information
- **Name:** TODO


---
# Part 0: Setup

Run this section first. It loads your API keys securely and creates the standard project folder structure.

In [ ]:
# Install any missing packages (uncomment if needed)
# !pip install openai plotly pandas numpy requests scipy

In [ ]:
from getpass import getpass
import os

# --- Load API keys securely (never hard-code keys in notebooks!) ---
if not os.environ.get("FRED_API_KEY"):
    fred_key = getpass("Enter your FRED API key (input hidden): ")
    if fred_key:
        os.environ["FRED_API_KEY"] = fred_key

if not os.environ.get("OPENAI_API_KEY"):
    openai_key = getpass("Enter your OpenAI API key (input hidden): ")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key

FRED_API_KEY    = os.environ.get("FRED_API_KEY", "")
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")

print("FRED key loaded:  ", bool(FRED_API_KEY))
print("OpenAI key loaded:", bool(OPENAI_API_KEY))

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
FOLDERS = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "output" / "figures",
    PROJECT_ROOT / "output" / "tables",
]
for p in FOLDERS:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
for p in FOLDERS:
    print(" -", p.relative_to(PROJECT_ROOT))

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.io as pio
from typing import Optional, Dict, Any, List
from openai import OpenAI

pio.renderers.default = "notebook_connected"

USE_REAL_LLM = bool(OPENAI_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY) if USE_REAL_LLM else None
print("OpenAI client ready:", USE_REAL_LLM)

---
# Part 1: Choose Your Research Question and Data (25 points)

You must investigate an economic question that is **different from the lecture demos**.

The lecture demos covered:
- Phillips Curve (unemployment + inflation)
- Oil prices + inflation
- Wages vs. inflation post-COVID
- Federal funds rate + inflation

You must choose something **not on that list**. Examples to spark ideas (pick your own!):
- How did housing starts respond to interest rate hikes after 2022?
- Is there a relationship between consumer sentiment and spending?
- Did the labor force participation rate recover after COVID as fast as employment did?
- How did the yield spread (10Y–2Y Treasury) behave before and after recent recessions?
- Is there a relationship between industrial production and oil prices?

---

## Question 1.1: State Your Research Question

**Research Question:**  
TODO — write your research question as a single clear sentence.

**Why it matters (2–4 sentences):**  
TODO — briefly explain why an economist, analyst, or policymaker would care about this question.

**Hypothesis / Prior (1–2 sentences):**  
Before looking at any data, what do you expect to find and why?  
TODO

## Question 1.2: Choose Your FRED Series

You must choose **at least 3 FRED series** to answer your question — none of which can be series already in the lecture's `APPROVED_SERIES` dictionary (`UNRATE`, `CPIAUCSL`, `GDP`, `GDPC1`, `DFF`, `DCOILWTICO`, `CES0500000003`, `PAYEMS`).

**How to find series:** Go to [fred.stlouisfed.org](https://fred.stlouisfed.org) and search for your topic. For each series you choose, record:

| Series ID | Full Name | Frequency | Units | Why relevant to your question |
|-----------|-----------|-----------|-------|--------------------------------|
| TODO | TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO | TODO |

> **Tip:** At least one of your series should have a **different frequency** from the others (e.g., one daily and one monthly, or one quarterly and one monthly). This forces you to handle frequency alignment, which is a real data engineering skill.

**Frequency mismatch note:** Which series have different frequencies? How will you align them?  
TODO

## Question 1.3: Build Your Data Toolkit

Below, implement (or adapt from lecture) the core data functions. **You must write these yourself** — you cannot paste the lecture code unchanged. At minimum you should adjust the functions to match your series, add error handling, or add one improvement of your own choice.

In [ ]:
# ── 1.3.A: FRED downloader ──────────────────────────────────────────────────
# TODO: implement or adapt fred_get_series()
# Must return a DataFrame with columns: [date, value, series_id]

def fred_get_series(series_id: str,
                    start_date: str = "2000-01-01",
                    end_date: Optional[str] = None,
                    api_key: Optional[str] = None) -> pd.DataFrame:
    """
    TODO: Add your own docstring describing this function.
    """
    # TODO: implement
    pass


# ── Quick test (run against your chosen series) ─────────────────────────────
# TODO: replace 'SERIES_ID_HERE' with one of your chosen series
test_df = fred_get_series("SERIES_ID_HERE", start_date="2005-01-01")
print(test_df.shape)
test_df.head()

In [ ]:
# ── 1.3.B: Merge multiple series into a wide DataFrame ──────────────────────
# TODO: implement merge_series_list()
# Must download all series in series_ids, align by date, and return a wide DataFrame

def merge_series_list(series_ids: List[str], start_date: str = "2000-01-01") -> pd.DataFrame:
    """
    TODO: Add your own docstring.
    """
    # TODO: implement
    pass


# ── 1.3.C: Frequency alignment ───────────────────────────────────────────────
# If any of your series have mismatched frequencies, write a function to handle it.
# If all series are the same frequency, write the function anyway and document
# that it is a no-op for your case.

def align_frequencies(df: pd.DataFrame, target_freq: str = "MS") -> pd.DataFrame:
    """
    Resample or forward-fill higher-frequency columns to match target_freq.
    target_freq: 'MS' = month-start, 'QS' = quarter-start, etc.
    
    TODO: Implement this for your specific series mix.
    If all your series are already monthly, document that here and return df unchanged.
    """
    # TODO: implement
    pass

In [ ]:
# ── 1.3.D: Analysis helpers ──────────────────────────────────────────────────
# Implement at least these three helpers. You may add more if your question requires it.

def yoy_pct_change(df: pd.DataFrame, col: str, periods: int = 12) -> pd.Series:
    """Year-over-year percent change. periods=12 for monthly, 4 for quarterly."""
    # TODO: implement
    pass


def corr_two(df: pd.DataFrame, col1: str, col2: str) -> float:
    """Pearson correlation between two columns, dropping NaN rows."""
    # TODO: implement
    pass


def plot_series_plotly(df: pd.DataFrame, y_cols: List[str], title: str = "") -> None:
    """Interactive Plotly line chart. df must have a 'date' column."""
    # TODO: implement
    pass

In [ ]:
# ── 1.3.E: Approved series dictionary ────────────────────────────────────────
# Build YOUR approved series dictionary using your chosen series.
# Format: "plain english concept" -> "FRED_SERIES_ID"
# You must include at least 3 NEW series (not in the lecture's dictionary).
# You may also include lecture series if they are relevant to your question.

MY_APPROVED_SERIES = {
    # TODO: populate with your own series
    # Example format:
    # "housing starts": "HOUST",
    # "30-year mortgage rate": "MORTGAGE30US",
    # "consumer sentiment": "UMCSENT",
}

print("Your approved concepts:", sorted(MY_APPROVED_SERIES.keys()))
print("Your FRED series IDs:  ", sorted(set(MY_APPROVED_SERIES.values())))

---
# Part 2: Add a New Tool (30 points)

The lecture agent had two tools: `build_dataset` and `analyze_dataset`. Your agent must include **one additional tool** that was not in lecture.

**Choose ONE of the following options:**

| Option | Tool to Build | Difficulty |
|--------|--------------|------------|
| **A** | Simple OLS regression tool | Intermediate |
| **B** | Rolling statistics tool (rolling mean, std, correlation) | Intermediate |
| **C** | Second data source integration (BLS or Census API) | Challenging |

Read the instructions for your chosen option below. **You only need to complete one.**

---

## Question 2.1: Which Option Did You Choose?

**My choice:** TODO (A, B, or C)

**Why I chose this option (1–2 sentences):**  
TODO

---
### Option A: OLS Regression Tool

Build a function `tool_run_regression(df, y_col, x_cols)` that:
1. Fits an OLS regression of `y_col` on `x_cols` using `numpy` (or `scipy.stats`)
2. Returns a dictionary with: coefficients, intercept, R², and a plain-English summary string
3. The agent's LLM planner must be able to request this tool when the user asks a question involving "relationship", "explain", "predict", or similar language

*Skip to cell 2.2 if you chose Option B or C.*

In [ ]:
# ── OPTION A: OLS Regression Tool ────────────────────────────────────────────
# Only complete this cell if you chose Option A.

from scipy import stats   # or use numpy.linalg.lstsq

def tool_run_regression(df: pd.DataFrame,
                        y_col: str,
                        x_cols: List[str]) -> Dict[str, Any]:
    """
    Run a simple OLS regression of y_col on x_cols.

    Returns a dict with:
      - 'coefficients': {col: float}
      - 'intercept': float
      - 'r_squared': float
      - 'n_obs': int
      - 'summary': plain-English interpretation string

    Handles NaN rows by dropping them before fitting.
    """
    # TODO: implement
    # Hint: use scipy.stats.linregress for simple (1 predictor) regression,
    # or numpy.linalg.lstsq for multiple predictors.
    pass


# ── Test your regression tool ────────────────────────────────────────────────
# TODO: test it on a sample of your data before wiring it into the agent
# Example (adapt to your columns):
# wide = merge_series_list([...], start_date="2005-01-01")
# reg_result = tool_run_regression(wide, y_col="YOUR_Y", x_cols=["YOUR_X"])
# print(reg_result)

---
### Option B: Rolling Statistics Tool

Build a function `tool_rolling_stats(df, col, windows)` that:
1. Computes rolling mean and rolling standard deviation for each window in `windows` (e.g., `[3, 6, 12]` months)
2. Adds these as new columns to the DataFrame
3. Plots the original series alongside its rolling means using Plotly
4. Returns the augmented DataFrame and a plain-English summary of what the rolling windows reveal

*Skip to cell 2.2 if you chose Option A or C.*

In [ ]:
# ── OPTION B: Rolling Statistics Tool ────────────────────────────────────────
# Only complete this cell if you chose Option B.

def tool_rolling_stats(df: pd.DataFrame,
                       col: str,
                       windows: List[int] = [3, 6, 12]) -> Dict[str, Any]:
    """
    Compute rolling mean and std for a series, plot all rolling means, 
    and return augmented DataFrame + summary.

    Returns a dict with:
      - 'df_augmented': original df with new rolling columns added
      - 'summary': plain-English string describing what the rolling windows show

    Hint: use df[col].rolling(window=w).mean() for rolling mean.
    """
    # TODO: implement
    pass


# ── Test your rolling stats tool ─────────────────────────────────────────────
# TODO: test before wiring into agent

---
### Option C: Second Data Source Integration (BLS or Census)

Build a function `tool_fetch_bls(series_ids, start_year, end_year)` **or** `tool_fetch_census(variables, state_fips)` that:
1. Fetches data from the BLS Public Data API or Census ACS API (both covered earlier in the course)
2. Returns a tidy DataFrame that can be **merged with a FRED DataFrame** by date (or by geography, for Census)
3. Documents any data quality or frequency alignment steps needed for the merge

**BLS API reference:** `https://api.bls.gov/publicAPI/v2/timeseries/data/`  
**Census ACS reference:** `https://api.census.gov/data/{year}/acs/acs5`

*Skip to cell 2.2 if you chose Option A or B.*

In [ ]:
# ── OPTION C: Second Data Source Integration ─────────────────────────────────
# Only complete this cell if you chose Option C.

def tool_fetch_bls(series_ids: List[str],
                   start_year: str,
                   end_year: str) -> pd.DataFrame:
    """
    Fetch one or more BLS series and return a tidy DataFrame.
    Columns: [date, value, series_id]
    """
    # TODO: implement
    # BLS API endpoint: https://api.bls.gov/publicAPI/v2/timeseries/data/
    # Registration is free and gives you a key with higher rate limits.
    pass


def merge_fred_and_bls(fred_df: pd.DataFrame,
                       bls_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge a FRED wide DataFrame and a BLS tidy DataFrame.
    Handle any frequency or date-format differences.
    """
    # TODO: implement
    pass


# ── Test your integration ────────────────────────────────────────────────────
# TODO: show a 5-row preview of the merged DataFrame

## Question 2.2: Verify Your New Tool Works

Run your new tool on at least one real example and display the output below. Then briefly explain what the output means.

In [ ]:
# TODO: call your new tool and display its output
# Example structure:
#   result = tool_run_regression(wide, y_col="...", x_cols=["..."])
#   print(result)

**What does the output tell you? (2–3 sentences):**  
TODO

---
# Part 3: Build and Run Your Agent (25 points)

Now assemble your agent. It must:
- Have a **planner** that calls the OpenAI API and returns a structured JSON plan
- Have an **executor** that routes the plan to the appropriate tool(s)
- Include **your new tool** from Part 2 alongside `build_dataset` and `analyze_dataset`
- Run on **three original queries** and display results

---

## Question 3.1: Write Your Planner System Instructions

Write the system prompt (instructions) that tells the LLM how to produce a structured plan for your agent. 

Your schema must include at minimum:
- `series_ids` — list of FRED series to fetch (from `MY_APPROVED_SERIES` only)
- `start_date` — date string
- `tasks` — list that can include `"plot"`, `"correlation"`, and your new tool's name
- `narrative` — 2–4 sentence interpretation plan in plain English

You may add extra schema fields if your new tool requires them.

In [ ]:
# ── 3.1: Planner system instructions ────────────────────────────────────────

MY_PLANNER_INSTRUCTIONS = f"""
TODO: Write your system prompt here.

It must tell the LLM:
  1. What format to return (JSON schema — define it clearly)
  2. Which series IDs are allowed (include MY_APPROVED_SERIES)
  3. When to invoke your new tool vs. a standard plot/correlation task
  4. Output ONLY valid JSON — no markdown, no extra text

Your approved series: {list(MY_APPROVED_SERIES.values())}
Your approved concepts: {list(MY_APPROVED_SERIES.keys())}
"""

print(MY_PLANNER_INSTRUCTIONS)

In [ ]:
# ── 3.2: Planner function ────────────────────────────────────────────────────

def my_llm_plan(question: str) -> Dict[str, Any]:
    """
    Send the user's question to the OpenAI LLM and return a structured plan.
    Falls back to a mock plan if USE_REAL_LLM is False.
    """
    prompt = f"User question: {question}"

    if USE_REAL_LLM:
        # TODO: call the OpenAI API using client.responses.create()
        # or client.chat.completions.create() — either works.
        # Parse the JSON from the response.
        pass
    else:
        # TODO: write a simple mock that returns a sensible default plan
        # using your MY_APPROVED_SERIES dictionary
        pass


# ── Quick test ───────────────────────────────────────────────────────────────
test_plan = my_llm_plan("What is the relationship between X and Y?")  # TODO: use a real question
print(json.dumps(test_plan, indent=2))

In [ ]:
# ── 3.3: Agent executor ──────────────────────────────────────────────────────

def my_economic_agent(question: str) -> Dict[str, Any]:
    """
    Full agent loop:
      1. Get a structured plan from the LLM planner
      2. Download and align data
      3. Execute requested tasks (plot, correlation, your new tool, etc.)
      4. Return a results dictionary

    Returns a dict with at minimum:
      - 'question'
      - 'plan'  (the raw JSON plan from the LLM)
      - 'series_ids_used'
      - 'analysis'  (dict of task outputs)
      - 'final_message'  (the narrative from the LLM plan)
    """
    # TODO: implement
    # Structure:
    #   plan = my_llm_plan(question)
    #   series_ids = validate and extract from plan
    #   df = merge_series_list(series_ids, start_date=plan['start_date'])
    #   df = align_frequencies(df)
    #   run tasks: plot, correlation, your new tool
    #   return results
    pass

## Question 3.2: Run Three Original Queries

Run your agent on **three different economic questions**. At least one must invoke your new tool from Part 2.

For each query:
- Show the agent's JSON plan
- Show the Plotly chart(s) produced
- Write a 3–5 sentence interpretation of what the results show

---

In [ ]:
# ── Query 1 ──────────────────────────────────────────────────────────────────
# Must be related to your research question from Part 1.

q1 = "TODO: write your first economic question here"
out1 = my_economic_agent(q1)

print("=== QUERY 1 ===")
print("Question:", out1["question"])
print("\nLLM Plan:")
print(json.dumps(out1["plan"], indent=2))
print("\nFinal Message:")
print(out1["final_message"])

**Query 1 — Your Interpretation (3–5 sentences):**  
TODO

In [ ]:
# ── Query 2 ──────────────────────────────────────────────────────────────────
# This query must trigger your new tool from Part 2.

q2 = "TODO: write a question that triggers your new tool"
out2 = my_economic_agent(q2)

print("=== QUERY 2 ===")
print("Question:", out2["question"])
print("\nLLM Plan:")
print(json.dumps(out2["plan"], indent=2))
print("\nFinal Message:")
print(out2["final_message"])

**Query 2 — Your Interpretation (3–5 sentences):**  
TODO

In [ ]:
# ── Query 3 ──────────────────────────────────────────────────────────────────
# Ask a question using a different subset of your approved series.

q3 = "TODO: write your third economic question here"
out3 = my_economic_agent(q3)

print("=== QUERY 3 ===")
print("Question:", out3["question"])
print("\nLLM Plan:")
print(json.dumps(out3["plan"], indent=2))
print("\nFinal Message:")
print(out3["final_message"])

**Query 3 — Your Interpretation (3–5 sentences):**  
TODO

---
# Part 4: Critical Evaluation (20 points)

A good data scientist doesn't just build a tool — they understand its limits. In this section you will deliberately probe the edges of your agent and reflect on what you find.

---

## Question 4.1: Try to Break Your Agent

Run your agent on each of the following "stress test" queries. Record what the agent does and what (if anything) goes wrong.

In [ ]:
# ── Stress Test A: Ambiguous concept ─────────────────────────────────────────
# Ask about a concept that is not in your approved series dictionary.
# Example: if your agent is about housing, ask about "economic anxiety"

stress_a = "TODO: write a question using a concept NOT in your approved dictionary"

try:
    result_a = my_economic_agent(stress_a)
    print("Plan returned:", json.dumps(result_a["plan"], indent=2))
    print("Series used:", result_a["series_ids_used"])
except Exception as e:
    print("Error:", e)

**What happened? (2–3 sentences):**  
TODO

In [ ]:
# ── Stress Test B: Contradictory or impossible request ───────────────────────
# Ask for something logically impossible or contradictory.
# Example: "Show me weekly data on GDP growth" (GDP is quarterly, not weekly)
# Or: "Compare all 10 of my series in a single correlation table"

stress_b = "TODO: write a contradictory or impossible request"

try:
    result_b = my_economic_agent(stress_b)
    print("Plan returned:", json.dumps(result_b["plan"], indent=2))
except Exception as e:
    print("Error:", e)

**What happened? (2–3 sentences):**  
TODO

In [ ]:
# ── Stress Test C: Your own stress test ──────────────────────────────────────
# Design ONE more stress test of your own that targets a specific weakness
# you suspect your agent has. Be creative.

stress_c = "TODO: write your own stress test"

try:
    result_c = my_economic_agent(stress_c)
    print("Plan returned:", json.dumps(result_c["plan"], indent=2))
except Exception as e:
    print("Error:", e)

**Why did you design this stress test? What did you find? (3–4 sentences):**  
TODO

## Question 4.2: Reflection Essay

Answer **all four** of the following in the Markdown cell below. Aim for 2–4 sentences per question.

1. **Where does your agent succeed?** Which type of queries does it handle well and why?

2. **Where does your agent fail or behave unexpectedly?** Describe at least two concrete failure modes you observed.

3. **How does your agent's behavior differ from what a human analyst would do?** Think specifically about how it selects series, interprets results, or handles edge cases.

4. **What is one meaningful enhancement you would add if you had more time?** Be specific — name the tool, feature, or data source you would add and explain how it would improve the agent.

**4.2 — Reflection:**

**1. Where it succeeds:**  
TODO

**2. Where it fails:**  
TODO

**3. How it differs from a human analyst:**  
TODO

**4. One meaningful enhancement:**  
TODO

---
# Submission Checklist

Before submitting, verify every item below is complete.

**Part 1**
- [ ] Research question clearly stated with hypothesis
- [ ] At least 3 NEW FRED series documented in the table (none from the lecture's dictionary)
- [ ] At least one series with a different frequency from the others
- [ ] `fred_get_series()`, `merge_series_list()`, `align_frequencies()`, `yoy_pct_change()`, `corr_two()`, `plot_series_plotly()` all implemented and tested
- [ ] `MY_APPROVED_SERIES` dictionary populated

**Part 2**
- [ ] One option (A, B, or C) fully implemented and tested
- [ ] New tool output shown and interpreted

**Part 3**
- [ ] System prompt written with a clear JSON schema
- [ ] Planner function calls the real OpenAI API (not just a mock)
- [ ] Agent executor routes to at least three tools
- [ ] Three queries run with plans displayed
- [ ] Plotly chart visible for each query
- [ ] 3–5 sentence interpretation written for each query

**Part 4**
- [ ] Three stress tests run and documented
- [ ] Reflection essay complete (all four questions answered)

**Final**
- [ ] Notebook runs top → bottom without errors
- [ ] No API keys hard-coded anywhere in the notebook
- [ ] Student name filled in at the top
- [ ] `.ipynb` file + PDF export both uploaded to Blackboard Ultra

---
# Grading Rubric

| Section | Points | What we look for |
|---------|--------|------------------|
| **Part 1: Data pipeline** | 25 | Correct FRED series (3+ new), frequency alignment handled, all helper functions work, approved dictionary populated |
| **Part 2: New tool** | 30 | Tool is correctly implemented, tested with real output, integrated into the agent executor, and actually invoked by at least one query |
| **Part 3: Agent + queries** | 25 | Planner uses real OpenAI API, executor routes correctly, three queries produce plans + plots + written interpretations |
| **Part 4: Critical evaluation** | 20 | Stress tests are thoughtful and documented, reflection essay shows genuine analysis (not generic) |
| **Total** | **100** | |

> **Professionalism deductions (up to −10):** Hard-coded API keys, notebook does not run top→bottom, no charts visible, or reflection answers are one sentence with no substance.